In [1]:
import pandas as pd
import os
import glob

BASE_DIR = "/Users/glossophilia/Library/CloudStorage/GoogleDrive-rumilofaniel@gmail.com/My Drive/Diss/Project Files/Project 1/Comps/Inputs"
BASE_FILE = "KD_annotations_350.csv"
OUTPUT_DIR = "/Users/glossophilia/Library/CloudStorage/GoogleDrive-rumilofaniel@gmail.com/My Drive/Diss/Project Files/Project 1/Comps/Results"

COLS = [
    "ASTrns", "ASUncc", "ASUnrg",
    "AktAcc", "AktAch", "AktAct", "AktSemel",
    "AktState",
    "ClCntfcl", "ClNegPol",
    "DtDTAM", "DtExp", "DtRes", "DtUniv",
    "TLAspPfv", "TLMdIrr", "TLTS",
    "VcPass",
]
KEY_COL = "TokenID"

os.makedirs(OUTPUT_DIR, exist_ok=True)

base = pd.read_csv(os.path.join(BASE_DIR, BASE_FILE)).set_index(KEY_COL)[COLS].sort_index()

compare_files = [
    f for f in glob.glob(os.path.join(BASE_DIR, "*.csv"))
    if os.path.basename(f) != BASE_FILE
]

print(f"Found {len(compare_files)} files to compare.\n")

rows = []

for comp_path in sorted(compare_files):
    comp_name = os.path.splitext(os.path.basename(comp_path))[0]
    comp = pd.read_csv(comp_path).set_index(KEY_COL)[COLS].sort_index()

    shared = base.index.intersection(comp.index)
    if len(shared) < len(base.index):
        print(f"  Note: {len(base.index) - len(shared)} base row(s) and {len(comp.index) - len(shared)} comp row(s) skipped (no matching TokenID)")
    base_aligned = base.loc[shared].fillna(0).astype(int)
    comp_aligned = comp.loc[shared].fillna(0).astype(int)

    mismatches = base_aligned != comp_aligned

    row = {"Run": comp_name}
    for c in COLS:
        row[c] = int(mismatches[c].sum())
        row[f"{c}_KD"] = int((mismatches[c] & (base_aligned[c] == 1)).sum())
    rows.append(row)
    print(f"Done: {comp_name}")

# Build summary DataFrame with interleaved columns (ASTrns, ASTrns_KD, ASUncc, ASUncc_KD, ...)
summary_cols = [col for c in COLS for col in (c, f"{c}_KD")]
summary = pd.DataFrame(rows).set_index("Run")[summary_cols]

out_path = os.path.join(OUTPUT_DIR, "mismatch_summary_claude_2a.csv")
summary.to_csv(out_path)
print(f"\nSummary saved to {out_path}")
summary

Found 7 files to compare.

  Note: 2 base row(s) and 2 comp row(s) skipped (no matching TokenID)
Done: Claude_abl1_ver
  Note: 2 base row(s) and 2 comp row(s) skipped (no matching TokenID)
Done: Claude_abl2_ver
  Note: 2 base row(s) and 2 comp row(s) skipped (no matching TokenID)
Done: Claude_abl3_ver
  Note: 2 base row(s) and 2 comp row(s) skipped (no matching TokenID)
Done: Claude_abl4_ver
  Note: 2 base row(s) and 2 comp row(s) skipped (no matching TokenID)
Done: Claude_abl5_ver
  Note: 2 base row(s) and 2 comp row(s) skipped (no matching TokenID)
Done: Claude_abl6_ver
  Note: 2 base row(s) and 2 comp row(s) skipped (no matching TokenID)
Done: Claude_full_ver

Summary saved to /Users/glossophilia/Library/CloudStorage/GoogleDrive-rumilofaniel@gmail.com/My Drive/Diss/Project Files/Project 1/Comps/Results/mismatch_summary_claude_2a.csv


,ASTrns,ASTrns_KD,ASUncc,ASUncc_KD,ASUnrg,ASUnrg_KD,AktAcc,AktAcc_KD,AktAch,AktAch_KD,...,DtUniv,DtUniv_KD,TLAspPfv,TLAspPfv_KD,TLMdIrr,TLMdIrr_KD,TLTS,TLTS_KD,VcPass,VcPass_KD
Run,,,,,,,,,,,,,,,,,,,,,
Claude_abl1_ver,38,14,20,15,22,12,86,53,87,9,...,35,28,69,59,16,5,71,1,6,3
Claude_abl2_ver,39,13,30,29,22,13,80,45,80,13,...,38,27,46,35,24,5,74,1,10,3
Claude_abl3_ver,37,19,24,19,26,12,89,57,78,20,...,37,32,80,64,17,6,75,1,8,2
Claude_abl4_ver,34,13,22,17,24,12,83,51,70,18,...,37,32,75,65,14,6,73,1,9,3
Claude_abl5_ver,38,15,23,19,19,11,86,65,82,18,...,35,29,84,76,22,4,72,1,6,3
Claude_abl6_ver,32,7,25,19,27,10,78,53,75,22,...,42,34,50,39,14,5,70,1,5,3
Claude_full_ver,35,12,21,16,22,12,86,57,79,16,...,37,30,71,61,16,5,71,1,7,3
